Gemma 4 E2B and E4B From Scratch (A Standalone Notebook)

In [ ]:
# 环境自检：从 importlib.metadata 读取已安装第三方库的版本号，
# 确认运行本 Gemma4 独立实现所需的依赖库版本是否满足要求。
from importlib.metadata import version

# 列出本notebook依赖的关键库：
# huggingface_hub 用于下载模型权重/分词器；safetensors 用于加载权重张量文件；
# tokenizers 用于分词；torch 用于搭建和运行模型本身。
pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "safetensors",      # to load the checkpoint tensors
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 选择要加载的 Gemma4 模型规模：
# "E2B"：约2B参数，emb_dim=1536，35层，1个KV头；
# "E4B"：约4B参数，emb_dim=2560，42层，2个KV头。
CHOOSE_MODEL = "E2B"  # Options: "E2B", "E4B"
# 是否使用指令微调(instruction-tuned, "-it")版本：
# True 时会从 google/gemma-4-{MODEL}-it 仓库下载权重，并使用对话模板(chat template)构造输入；
# False 时使用基础(base)模型权重，用简单的问答格式拼接输入。
USE_INSTRUCT_MODEL = True

1. Architecture code

In [ ]:
# ===== 第1部分：模型结构定义 =====
# 本cell实现 Gemma4 稠密(dense)模型的核心组件：
#   1) RoPE（旋转位置编码）参数计算与应用；
#   2) Gemma4RMSNorm（均方根归一化，比LayerNorm省去减均值步骤）；
#   3) Gemma4FeedForward（GeGLU门控前馈网络）；
#   4) Gemma4Attention（分组查询注意力GQA，支持局部滑窗/全局两种注意力类型，
#      并支持跨层KV共享以节省计算/显存）；
#   5) Gemma4DenseBlock（单层Transformer Block，含Pre/Post双重归一化与per-layer输入注入）；
#   6) Gemma4DenseModel（完整模型：embedding -> N层block -> 最终norm -> 输出头）。
import torch
import torch.nn as nn


# ---- RoPE（旋转位置编码）参数计算 ----
# 根据 head_dim 与旋转基数 theta_base，预先计算出每个位置、每个频率分量的 cos/sin 查找表。
# Gemma4 对局部滑窗层和全局层使用不同的RoPE配置：
#   - 局部层：rope_type="default"，theta_base 较小（如1万），对全部维度旋转；
#   - 全局层：rope_type="proportional"，theta_base 较大（如100万），且只对头维度的
#     一部分(partial_rotary_factor)做旋转，其余维度角度置0（即NoPE，不编码位置信息）。
def compute_rope_params(
    head_dim,
    theta_base=10_000.0,
    context_length=4096,
    rope_type="default",
    partial_rotary_factor=1.0,
    dtype=torch.float32,
):
    # proportional 模式：仅对 head_dim 的前 partial_rotary_factor 比例的维度施加旋转，
    # 其余维度的 inv_freq 置0（NoPE），rope_angles 是参与旋转的频率分量个数。
    if rope_type == "proportional":
        rope_angles = int(partial_rotary_factor * head_dim // 2)
        inv_freq_rotated = 1.0 / (
            theta_base ** (torch.arange(0, 2 * rope_angles, 2, dtype=torch.float32) / head_dim)
        )
        nope_angles = head_dim // 2 - rope_angles
        if nope_angles > 0:
            inv_freq = torch.cat([inv_freq_rotated, torch.zeros(nope_angles, dtype=torch.float32)], dim=0)
        else:
            inv_freq = inv_freq_rotated
    else:
        # default 模式：标准 RoPE，对 head_dim 的全部偶数维计算逆频率 1/theta_base^(2i/head_dim)。
        inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))

    # 生成位置索引 [0, 1, ..., context_length-1]，形状 (context_length,)。
    positions = torch.arange(context_length, dtype=torch.float32)
    # 外积得到每个位置、每个频率分量的角度，形状 (context_length, head_dim//2)。
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
    # 将角度复制拼接一份，形状变为 (context_length, head_dim)，
    # 对应 apply_rope 中把向量切成前后两半分别旋转的 rotate_half 技巧。
    angles = torch.cat([angles, angles], dim=1)
    # 预先算好 cos/sin 查找表(float32)，推理时直接按位置切片使用，避免重复计算。
    cos = torch.cos(angles).to(dtype)
    sin = torch.sin(angles).to(dtype)
    return cos, sin


# ---- 应用 RoPE：把旋转位置编码乘到 Q/K 张量上 ----
# x 形状: (batch_size, num_heads, seq_len, head_dim)。
def apply_rope(x, cos, sin):
    batch_size, num_heads, seq_len, head_dim = x.shape
    # head_dim 必须为偶数，因为下面要把它从中间切成两半做 rotate_half。
    assert head_dim % 2 == 0, "Head dimension must be even"

    # 把最后一维切成前后两半：x1 对应前半段 [0, head_dim/2)，x2 对应后半段 [head_dim/2, head_dim)。
    x1 = x[..., : head_dim // 2]
    x2 = x[..., head_dim // 2 :]

    # 取出当前序列长度对应的 cos/sin，并广播出 batch 与 head 维度：
    # (seq_len, head_dim) -> (1, 1, seq_len, head_dim)。
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)
    # rotate_half：把后半段取负号放到前面、前半段放到后面，
    # 这样和cos/sin相乘相加后，等价于对(x1, x2)做复数旋转 (x1+i*x2)*e^{i*theta}。
    rotated = torch.cat((-x2, x1), dim=-1)
    # RoPE 核心公式：x_rotated = x*cos + rotate_half(x)*sin；最后转回输入原始dtype。
    return ((x * cos) + (rotated * sin)).to(dtype=x.dtype)


# ---- KV 头广播：分组查询注意力(GQA)中，把K/V的头重复到与Q头数量相同 ----
# x: (batch_size, num_kv_heads, seq_len, head_dim)
# -> (batch_size, num_kv_heads*repeats, seq_len, head_dim)，repeats = num_heads // num_kv_heads。
def repeat_kv(x, repeats):
    if repeats == 1:
        return x
    return x.repeat_interleave(repeats, dim=1)

# ---- RMSNorm：Gemma系列使用的均方根归一化 ----
# 与LayerNorm相比不做去均值(减mean)，只用均方根(RMS)做缩放，计算更简单。
# 公式：x_norm = x / sqrt(mean(x^2, dim=-1) + eps)，若 with_scale=True 再乘以可学习权重 weight。
# v_norm 使用 with_scale=False（不做可学习缩放），仅对Value做数值稳定化处理。
class Gemma4RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6, with_scale=True):
        super().__init__()
        self.eps = eps
        self.with_scale = with_scale
        if with_scale:
            self.weight = nn.Parameter(torch.ones(dim))

    # 用float32计算归一化过程以保证数值稳定，即使输入是bfloat16/float16。
    def forward(self, x):
        x_float = x.float()
        # 沿最后一维(特征维)计算均方值，再加eps防止除零。
        mean_squared = x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps
        x_norm = x_float * torch.pow(mean_squared, -0.5)
        if self.with_scale:
            x_norm = x_norm * self.weight.float()
        # 计算完成后转换回输入原始的dtype（如bfloat16），减少精度累计误差同时节省显存。
        return x_norm.to(dtype=x.dtype)



# ---- 门控前馈网络(Gated MLP / GeGLU变体) ----
# Gemma4 使用GELU门控：down_proj(gelu(gate_proj(x)) * up_proj(x))。
# 若当前层属于"KV共享层"且开启 use_double_wide_mlp，则中间层宽度翻倍(intermediate_size*2)，
# 用于补偿因跳过独立K/V投影而节省下来的计算量。
class Gemma4FeedForward(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        # 计算从第几层开始是"KV共享层"：总层数 - 共享层数量。
        # 共享层会直接复用前面同类型层算出的K/V，不再自己做K/V投影，从而节省显存和计算。
        first_kv_shared_layer_idx = cfg["n_layers"] - cfg["num_kv_shared_layers"]
        is_kv_shared_layer = layer_idx >= first_kv_shared_layer_idx > 0
        use_double_wide_mlp = cfg["use_double_wide_mlp"] and is_kv_shared_layer
        intermediate_size = cfg["hidden_dim"] * (2 if use_double_wide_mlp else 1)
        self.gate_proj = nn.Linear(cfg["emb_dim"], intermediate_size, bias=False, dtype=cfg["dtype"])
        self.up_proj = nn.Linear(cfg["emb_dim"], intermediate_size, bias=False, dtype=cfg["dtype"])
        self.down_proj = nn.Linear(intermediate_size, cfg["emb_dim"], bias=False, dtype=cfg["dtype"])

    # x: (batch_size, seq_len, emb_dim) -> 输出同形状。
    def forward(self, x):
        x_gate = self.gate_proj(x)
        x_up = self.up_proj(x)
        # 用tanh近似的GELU作为门控，逐元素乘以up_proj分支，构成GeGLU结构。
        x = nn.functional.gelu(x_gate, approximate="tanh") * x_up
        return self.down_proj(x)

# ---- 分组查询注意力(GQA)，支持局部滑窗/全局两种模式，以及跨层KV共享 ----
# Gemma4 的注意力层按 layer_types 交替使用两种类型：
#   - "sliding_attention"(局部注意力)：只能看到最近 sliding_window 个token，
#     使用较小的 head_dim，RoPE base 较小(默认1万)；
#   - "full_attention"(全局注意力)：可看到全部历史token(标准因果掩码)，
#     使用较大的 head_dim(global_head_dim)，RoPE base 较大(proportional类型，约100万)。
# 此外还支持跨层KV共享：后面若干层可直接复用更早的同类型层算出的K/V，减少重复计算。
class Gemma4Attention(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        # 根据当前层号从 cfg["layer_types"] 查出该层是局部滑窗还是全局注意力。
        self.layer_type = cfg["layer_types"][layer_idx]
        self.is_sliding = self.layer_type == "sliding_attention"
        # 局部层用较小的 head_dim，全局层用较大的 global_head_dim ——
        # Gemma4的一个特点：不同类型的层可以有不同的注意力头维度。
        self.head_dim = cfg["head_dim"] if self.is_sliding else cfg["global_head_dim"]
        self.num_heads = cfg["n_heads"]
        self.num_kv_heads = cfg["n_kv_heads"]
        # GQA分组数：每个KV头要被多少个Q头共享，例如 num_heads=8, num_kv_heads=1 时为8。
        self.num_key_value_groups = self.num_heads // self.num_kv_heads
        # Q/K/V/O 四个线性投影，均无bias。K/V的输出维度用 num_kv_heads（远小于num_heads），
        # 这正是GQA节省参数/显存的关键所在。
        self.q_proj = nn.Linear(cfg["emb_dim"], self.num_heads * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.k_proj = nn.Linear(cfg["emb_dim"], self.num_kv_heads * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.v_proj = nn.Linear(cfg["emb_dim"], self.num_kv_heads * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, cfg["emb_dim"], bias=False, dtype=cfg["dtype"])
        # Gemma4 对Q、K各自做一次RMSNorm(QK-Norm)，有助于训练/推理数值稳定；
        # v_norm 对V做归一化但不带可学习缩放(with_scale=False)。
        self.q_norm = Gemma4RMSNorm(self.head_dim, eps=cfg["layer_norm_eps"])
        self.k_norm = Gemma4RMSNorm(self.head_dim, eps=cfg["layer_norm_eps"])
        self.v_norm = Gemma4RMSNorm(self.head_dim, eps=cfg["layer_norm_eps"], with_scale=False)

        # 判断当前层是否属于"KV共享层"：若是，则该层forward时不再自己计算K/V，
        # 而是复用 kv_shared_layer_index 指向的、更早的同类型层缓存下来的K/V。
        first_kv_shared_layer_idx = cfg["n_layers"] - cfg["num_kv_shared_layers"]
        self.is_kv_shared_layer = layer_idx >= first_kv_shared_layer_idx > 0
        prev_layers = cfg["layer_types"][:first_kv_shared_layer_idx]
        if self.is_kv_shared_layer:
            # 在"共享起点"之前的层里，找到最近一个同类型(layer_type)的层，记下其层号，
            # 作为本层将要复用的K/V来源。
            self.kv_shared_layer_index = len(prev_layers) - 1 - prev_layers[::-1].index(self.layer_type)
            self.store_full_length_kv = False
        else:
            self.kv_shared_layer_index = None
            # 若本层不是共享层，但它是"共享起点"之前最后一个同类型层，
            # 则需要把自己算出的K/V保留下来(store_full_length_kv=True)，供后面的共享层复用。
            self.store_full_length_kv = (
                first_kv_shared_layer_idx > 0
                and self.layer_type in prev_layers
                and layer_idx == len(prev_layers) - 1 - prev_layers[::-1].index(self.layer_type)
            )

    # x: (batch_size, seq_len, emb_dim)；
    # mask: (seq_len, seq_len) 布尔张量，True表示该位置要被屏蔽(不可见)；
    # cos, sin: 对应本层类型(局部/全局)的RoPE查找表；
    # shared_kv: 若非None，直接复用传入的(key, value)而不重新计算；
    # 返回 (output, computed_kv)：computed_kv 仅在 return_kv=True 且本层自行计算K/V时返回，
    # 用于提供给后续的共享层使用。
    def forward(self, x, mask, cos, sin, shared_kv=None, return_kv=False):
        batch_size, seq_len, _ = x.shape
        # 线性投影后 reshape 成多头形式：
        # (batch, seq_len, num_heads*head_dim) -> (batch, seq_len, num_heads, head_dim)
        # -> transpose(1,2) -> (batch, num_heads, seq_len, head_dim)。
        query = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        # 对Q做RMSNorm(QK-Norm)。
        query = self.q_norm(query)
        # 对Q施加RoPE旋转位置编码。
        query = apply_rope(query, cos, sin)

        computed_kv = None
        # 只有当没有传入共享K/V时，才自己计算K/V；否则直接复用shared_kv，跳过下面的投影计算。
        if shared_kv is None:
            # K/V同样reshape成多头形式，但头数是 num_kv_heads（远小于num_heads）。
            key = self.k_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
            value = self.v_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
            key = self.k_norm(key)
            value = self.v_norm(value)
            # 只对K施加RoPE，V不需要位置编码。
            key = apply_rope(key, cos, sin)
            # 记录本层自己算出的K/V，供 return_kv=True 时返回，供后续共享层使用。
            computed_kv = (key, value)
        else:
            # 直接复用调用方传入的、之前某层算好的K/V（跨层KV共享，节省计算与显存）。
            key, value = shared_kv

        # GQA关键步骤：把K/V在头维度上重复 num_key_value_groups 次，使其头数与Q对齐，
        # 这样才能做逐头的注意力矩阵乘法。
        key_for_attn = repeat_kv(key, self.num_key_value_groups)
        value_for_attn = repeat_kv(value, self.num_key_value_groups)

        # 计算注意力分数：
        # (batch, num_heads, seq_len_q, head_dim) @ (batch, num_heads, head_dim, seq_len_k)
        # -> (batch, num_heads, seq_len_q, seq_len_k)。
        attn_scores = query @ key_for_attn.transpose(-1, -2)
        # 用因果/滑窗掩码把不可见位置的分数填成该dtype能表示的最小值，softmax后趋近于0。
        # 注意这里没有除以 sqrt(head_dim) 做缩放——Gemma4通过QK-Norm来稳定数值，替代了传统的attention scaling。
        attn_scores = attn_scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), torch.finfo(attn_scores.dtype).min)
        # 用float32计算softmax以保证数值稳定，再转换回Q原本的dtype。
        attn_weights = torch.softmax(attn_scores.float(), dim=-1).to(dtype=query.dtype)
        # 加权求和得到上下文向量：
        # (batch, num_heads, seq_len_q, seq_len_k) @ (batch, num_heads, seq_len_k, head_dim)
        # -> (batch, num_heads, seq_len_q, head_dim)。
        context = attn_weights @ value_for_attn
        # 把多头结果合并回单一向量：
        # (batch, num_heads, seq_len, head_dim) -> (batch, seq_len, num_heads*head_dim)。
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.num_heads * self.head_dim)
        # 输出投影，映射回 emb_dim。
        output = self.o_proj(context)
        # 若需要缓存K/V给后续共享层使用则一并返回；否则第二个返回值为None。
        if return_kv and computed_kv is not None:
            return output, computed_kv
        return output, None

# ---- Transformer Block：注意力子层 + 前馈子层，外加Gemma4特有的"每层输入"机制 ----
# 结构采用 Pre-Norm + Post-Norm 双重归一化
# (input_layernorm/post_attention_layernorm, pre_feedforward_layernorm/post_feedforward_layernorm)，
# 比单纯Pre-Norm在数值上更稳定。
class Gemma4DenseBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        self.layer_idx = layer_idx
        self.layer_type = cfg["layer_types"][layer_idx]
        self.att = Gemma4Attention(cfg, layer_idx)
        self.mlp = Gemma4FeedForward(cfg, layer_idx)
        self.input_layernorm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.post_attention_layernorm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.pre_feedforward_layernorm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.post_feedforward_layernorm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        # layer_scalar 是一个可从权重加载的标量缩放系数(默认初始化为1.0)，作用在该block最终输出上。
        self.register_buffer("layer_scalar", torch.ones(1), persistent=True)
        self.hidden_size_per_layer_input = cfg["hidden_size_per_layer_input"]
        # Gemma4的"每层输入"(per-layer input)机制：为每一层准备一份从输入token id
        # 单独embedding出来的小向量，以门控方式注入该层的残差流中，
        # 用于在不显著增加主干宽度的情况下提升模型表达能力。
        if self.hidden_size_per_layer_input:
            self.per_layer_input_gate = nn.Linear(
                cfg["emb_dim"],
                self.hidden_size_per_layer_input,
                bias=False,
                dtype=cfg["dtype"],
            )
            self.per_layer_projection = nn.Linear(
                self.hidden_size_per_layer_input,
                cfg["emb_dim"],
                bias=False,
                dtype=cfg["dtype"],
            )
            self.post_per_layer_input_norm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])

    # x: (batch, seq_len, emb_dim)；
    # per_layer_input: 该层专属的per-layer embedding，形状(batch, seq_len, hidden_size_per_layer_input)；
    # mask_local/mask_global, cos_local/sin_local, cos_global/sin_global：
    # 局部滑窗层和全局层分别使用各自的掩码与RoPE表。
    def forward(
        self,
        x,
        per_layer_input,
        mask_local,
        mask_global,
        cos_local,
        sin_local,
        cos_global,
        sin_global,
        shared_kv=None,
        return_kv=False,
    ):
        # 根据本层是sliding_attention还是full_attention，选用对应的掩码与RoPE参数。
        mask = mask_local if self.layer_type == "sliding_attention" else mask_global
        cos = cos_local if self.layer_type == "sliding_attention" else cos_global
        sin = sin_local if self.layer_type == "sliding_attention" else sin_global

        # ---- 注意力子层：Pre-Norm -> Attention -> Post-Norm -> 残差相加 ----
        residual = x
        x = self.input_layernorm(x)
        x_attn, cached_kv = self.att(x, mask, cos, sin, shared_kv=shared_kv, return_kv=return_kv)
        x_attn = self.post_attention_layernorm(x_attn)
        x = residual + x_attn

        # ---- 前馈子层：Pre-Norm -> MLP -> Post-Norm -> 残差相加 ----
        residual = x
        x = self.pre_feedforward_layernorm(x)
        x = self.mlp(x)
        x = self.post_feedforward_layernorm(x)
        x = residual + x

        # ---- 每层输入注入：门控后与主干残差流融合 ----
        if self.hidden_size_per_layer_input:
            residual = x
            x_per_layer = self.per_layer_input_gate(x)
            x_per_layer = nn.functional.gelu(x_per_layer, approximate="tanh")
            x_per_layer = x_per_layer * per_layer_input
            x_per_layer = self.per_layer_projection(x_per_layer)
            x_per_layer = self.post_per_layer_input_norm(x_per_layer)
            x = residual + x_per_layer

        # 最终乘以layer_scalar做整体缩放后返回，同时把本层算出的K/V(cached_kv)透传给上层调用者。
        return x * self.layer_scalar.to(dtype=x.dtype), cached_kv

# ---- 整个Gemma4稠密模型：Embedding -> N层Transformer Block -> 最终Norm -> 输出头 ----
class Gemma4DenseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["layer_types"] is not None and len(cfg["layer_types"]) == cfg["n_layers"]
        self.cfg = cfg
        # 词嵌入表：(vocab_size, emb_dim)。
        self.tok_emb = nn.Embedding(
            cfg["vocab_size"],
            cfg["emb_dim"],
            padding_idx=cfg.get("pad_token_id", 0),
            dtype=cfg["dtype"],
        )
        # 堆叠 n_layers 个 Transformer Block，每个block内部会根据layer_idx自行决定
        # 自己是局部滑窗层/全局层，以及是否是KV共享层。
        self.blocks = nn.ModuleList([Gemma4DenseBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.final_norm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        # 输出头：把最终隐藏状态映射回词表大小的logits。
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])
        if cfg.get("tie_word_embeddings", False):
            # 权重绑定(tie embedding)：输出头与输入embedding共享同一份权重矩阵，减少参数量。
            self.out_head.weight = self.tok_emb.weight

        self.hidden_size_per_layer_input = cfg["hidden_size_per_layer_input"]
        # 每层输入的embedding表：为每一层准备一份专属的小型embedding，
        # 总输出维度是 n_layers * hidden_size_per_layer_input，之后按层切片使用。
        if self.hidden_size_per_layer_input:
            self.embed_tokens_per_layer = nn.Embedding(
                cfg["vocab_size_per_layer_input"],
                cfg["n_layers"] * self.hidden_size_per_layer_input,
                padding_idx=cfg.get("pad_token_id", 0),
                dtype=cfg["dtype"],
            )
            self.per_layer_model_projection = nn.Linear(
                cfg["emb_dim"],
                cfg["n_layers"] * self.hidden_size_per_layer_input,
                bias=False,
                dtype=cfg["dtype"],
            )
            self.per_layer_projection_norm = Gemma4RMSNorm(
                self.hidden_size_per_layer_input,
                eps=cfg["layer_norm_eps"],
            )

        # 分别为"局部滑窗层"和"全局层"预计算各自的RoPE cos/sin表：
        # 局部层用较小base、对全部维度旋转；全局层用较大base，且只旋转部分维度(NoPE + proportional)。
        rope_local_type = cfg.get("rope_local_type", "default")
        cos_local, sin_local = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_local_base"],
            context_length=cfg["context_length"],
            rope_type=rope_local_type,
            dtype=torch.float32,
        )
        cos_global, sin_global = compute_rope_params(
            head_dim=cfg["global_head_dim"],
            theta_base=cfg["rope_global_base"],
            context_length=cfg["context_length"],
            rope_type=cfg["rope_global_type"],
            partial_rotary_factor=cfg["rope_global_partial_rotary_factor"],
            dtype=torch.float32,
        )
        # 用register_buffer注册为非persistent buffer：会随模型.to(device)移动，
        # 但不会被保存进state_dict/checkpoint(因为可以按需重新计算)。
        self.register_buffer("cos_local", cos_local, persistent=False)
        self.register_buffer("sin_local", sin_local, persistent=False)
        self.register_buffer("cos_global", cos_global, persistent=False)
        self.register_buffer("sin_global", sin_global, persistent=False)

    # 构造两种因果注意力掩码：
    #   mask_global：标准下三角因果掩码（可看到自己和之前所有token）；
    #   mask_local ：在因果掩码基础上，再屏蔽掉超出 sliding_window 范围的"太久远"的token，
    #                实现局部滑窗注意力。
    # 掩码中 True 表示该位置要被屏蔽(不可见)。
    def _create_masks(self, seq_len, device):
        ones = torch.ones((seq_len, seq_len), dtype=torch.bool, device=device)
        # 上三角(不含对角线)为True，即屏蔽"未来"的token —— 标准因果掩码。
        mask_global = torch.triu(ones, diagonal=1)
        # 构造"过于久远"的掩码：先取以sliding_window为偏移的上三角，再转置，
        # 得到距离超过sliding_window的位置为True。
        far_past = torch.triu(ones, diagonal=self.cfg["sliding_window"]).T
        # 局部掩码 = 因果掩码 或 过于久远掩码，两者取并集(满足任一条件即屏蔽)。
        mask_local = mask_global | far_past
        return mask_global, mask_local

    # 从input_ids查出每层专属的per-layer embedding，按sqrt(hidden_size_per_layer_input)缩放
    # (类似Transformer对embedding的缩放习惯)，再reshape成按层切片的形状：
    # (batch, seq_len, n_layers, hidden_size_per_layer_input)。
    def get_per_layer_inputs(self, input_ids):
        if not self.hidden_size_per_layer_input:
            return None
        return (self.embed_tokens_per_layer(input_ids) * (self.hidden_size_per_layer_input ** 0.5)).reshape(
            *input_ids.shape,
            self.cfg["n_layers"],
            self.hidden_size_per_layer_input,
        )

    # 把主干隐藏状态(inputs_embeds)投影到每一层的per-layer空间，
    # 并与get_per_layer_inputs算出的embedding做融合(若提供)，再做RMSNorm。
    # 这是Gemma4用主干信息"重新校准"per-layer embedding的机制。
    def project_per_layer_inputs(self, inputs_embeds, per_layer_inputs=None):
        if not self.hidden_size_per_layer_input:
            return None
        projected = self.per_layer_model_projection(inputs_embeds) * (self.cfg["emb_dim"] ** -0.5)
        projected = projected.reshape(
            *inputs_embeds.shape[:-1],
            self.cfg["n_layers"],
            self.hidden_size_per_layer_input,
        )
        projected = self.per_layer_projection_norm(projected)
        if per_layer_inputs is None:
            return projected
        # 两路信号相加后乘以 1/sqrt(2) 做归一化，保持方差稳定(类似残差缩放技巧)。
        return (projected + per_layer_inputs) * (2.0 ** -0.5)

    # 整体前向传播：
    # input_ids: (batch_size, seq_len) token id；
    # reuse_shared_kv: 是否启用跨层KV共享（生成时通常开启以减少重复计算）；
    # 返回 logits: (batch_size, seq_len, vocab_size)。
    def forward(self, input_ids, reuse_shared_kv=False):
        # 词嵌入后按 sqrt(emb_dim) 缩放，是Gemma系列的标准做法，
        # 用来平衡embedding与后续RMSNorm/残差流之间的数值尺度。
        x = self.tok_emb(input_ids) * (self.cfg["emb_dim"] ** 0.5)
        per_layer_inputs = self.get_per_layer_inputs(input_ids)
        per_layer_inputs = self.project_per_layer_inputs(x, per_layer_inputs)
        # 根据当前序列长度构造这一次前向要用的两种掩码。
        mask_global, mask_local = self._create_masks(input_ids.size(1), input_ids.device)
        # 用一个字典缓存"被跨层共享的K/V"，key为提供K/V的层号。
        shared_layer_kv = {} if reuse_shared_kv else None

        # 逐层执行Transformer Block。
        for i, block in enumerate(self.blocks):
            # 取出该层专属的per-layer输入切片: (batch, seq_len, hidden_size_per_layer_input)。
            per_layer_input = per_layer_inputs[:, :, i, :] if per_layer_inputs is not None else None
            shared_kv = None
            need_store_kv = False
            # 如果启用KV共享：
            #   - 若本层是共享层，去shared_layer_kv里取它应复用的K/V；
            #   - 若本层是"共享起点"前最后一个同类型层，需把自己算出的K/V存下来(need_store_kv)。
            if reuse_shared_kv:
                shared_kv = shared_layer_kv.get(block.att.kv_shared_layer_index) if block.att.is_kv_shared_layer else None
                need_store_kv = block.att.store_full_length_kv and shared_kv is None
            x, cached_kv = block(
                x,
                per_layer_input,
                mask_local,
                mask_global,
                self.cos_local,
                self.sin_local,
                self.cos_global,
                self.sin_global,
                shared_kv=shared_kv,
                return_kv=need_store_kv,
            )
            # 把本层算出的K/V存入共享缓存，供后面的共享层在同一次forward中复用。
            if reuse_shared_kv and need_store_kv and cached_kv is not None:
                shared_layer_kv[i] = cached_kv

        # 所有层结束后做最终RMSNorm，再过输出头得到logits。
        x = self.final_norm(x)
        logits = self.out_head(x)
        # logit软上限(soft-capping)：将logits压缩到[-final_logit_softcap, final_logit_softcap]
        # 区间内，用tanh做平滑饱和，防止极端logit值，提升训练/推理稳定性。
        if self.cfg.get("final_logit_softcap") is not None:
            logits = logits / self.cfg["final_logit_softcap"]
            logits = torch.tanh(logits)
            logits = logits * self.cfg["final_logit_softcap"]
        return logits



# 别名：本notebook只实现了稠密(dense)版本，因此Gemma4Model直接指向Gemma4DenseModel
# （区别于MoE版本的Gemma4，那种版本会包含专家路由结构）。
Gemma4Model = Gemma4DenseModel

2. Initialize model

In [ ]:
# ===== 第2部分：初始化模型 =====
# 定义Gemma4 E2B / E4B两种规模的超参数配置字典。关键点：
#   - head_dim(局部层) vs global_head_dim(全局层)：两种注意力头维度不同；
#   - n_kv_heads 远小于 n_heads：分组查询注意力(GQA)；
#   - layer_types：局部滑窗层与全局层的交替模式；
#   - num_kv_shared_layers：末尾若干层复用更早层的K/V(跨层KV共享)；
#   - rope_local_* vs rope_global_*：两套不同base/类型的RoPE参数。
def get_gemma4_dense_config(model_size="E2B", dtype=torch.bfloat16):
    model_size = model_size.upper()

    if model_size == "E2B":
        # ---- E2B配置：约2B参数量 ----
        return {
            "vocab_size": 262_144,
            "vocab_size_per_layer_input": 262_144,
            "emb_dim": 1536,
            "hidden_dim": 4 * 1536,
            "n_layers": 35,
            # 注意力头配置：8个Q头，但只有1个KV头(接近极致的GQA/MQA)；
            # 局部层头维度256，全局层头维度512(更大)。
            "n_heads": 8,
            "head_dim": 256,
            "n_kv_heads": 1,
            "num_global_kv_heads": None,
            "global_head_dim": 512,
            # 支持长达131072 tokens的上下文；滑窗层只关注最近512个token。
            "context_length": 131_072,
            "sliding_window": 512,
            # 每5层为一组：4层局部滑窗 + 1层全局注意力，如此重复7组共35层。
            "layer_types": (["sliding_attention"] * 4 + ["full_attention"]) * 7,
            "hidden_size_per_layer_input": 256,
            # 每层都有独立的per-layer embedding输入(256维)；
            # 最后20层属于KV共享层，复用更早层的K/V以节省计算。
            "num_kv_shared_layers": 20,
            "use_double_wide_mlp": True,
            "attention_k_eq_v": False,
            # 局部层RoPE: base=1万，对全部维度旋转；全局层RoPE: base=100万，
            # 且只旋转25%的维度(partial_rotary_factor)，其余维度不编码位置信息(NoPE)。
            "rope_local_base": 10_000.0,
            "rope_local_type": "default",
            "rope_global_base": 1_000_000.0,
            "rope_global_type": "proportional",
            "rope_global_partial_rotary_factor": 0.25,
            "layer_norm_eps": 1e-6,
            # 输出logits会被压缩到±30范围内(soft-capping)，且输出层与输入embedding权重绑定。
            "final_logit_softcap": 30.0,
            "tie_word_embeddings": True,
            "pad_token_id": 0,
            "dtype": dtype,
        }

    # ---- E4B配置：约4B参数量。相比E2B：emb_dim更大(2560)、层数更多(42层)、
    # 2个KV头、局部:全局层比例变为5:1、KV共享层数18、且不使用双倍宽MLP ----
    if model_size == "E4B":
        return {
            "vocab_size": 262_144,
            "vocab_size_per_layer_input": 262_144,
            "emb_dim": 2560,
            "hidden_dim": 4 * 2560,
            "n_layers": 42,
            "n_heads": 8,
            "head_dim": 256,
            "n_kv_heads": 2,
            "num_global_kv_heads": None,
            "global_head_dim": 512,
            "context_length": 131_072,
            "sliding_window": 512,
            "layer_types": (["sliding_attention"] * 5 + ["full_attention"]) * 7,
            "hidden_size_per_layer_input": 256,
            "num_kv_shared_layers": 18,
            "use_double_wide_mlp": False,
            "attention_k_eq_v": False,
            "rope_local_base": 10_000.0,
            "rope_local_type": "default",
            "rope_global_base": 1_000_000.0,
            "rope_global_type": "proportional",
            "rope_global_partial_rotary_factor": 0.25,
            "layer_norm_eps": 1e-6,
            "final_logit_softcap": 30.0,
            "tie_word_embeddings": True,
            "pad_token_id": 0,
            "dtype": dtype,
        }

    raise ValueError(f"Unknown Gemma 4 dense size: {model_size}")
# ---- 自动选择运行设备与推理精度 ----
# GPU上用bfloat16以节省显存/提速；MPS(Apple Silicon)和CPU上退回float32以保证兼容性/精度。
if torch.cuda.is_available():
    device = torch.device("cuda")
    model_dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    model_dtype = torch.float32
else:
    device = torch.device("cpu")
    model_dtype = torch.float32

# 这里先用float32构建一次配置仅用于打印/检查模型规模参数，真正建模型时会用model_dtype。
selected_cfg = get_gemma4_dense_config(CHOOSE_MODEL, dtype=torch.float32)
# 以下是一个未赋值的字典表达式，在Jupyter中会被当作该cell的输出直接显示，
# 用于快速查看当前选择的模型的关键结构参数与运行设备/精度。
{
    "model": CHOOSE_MODEL,
    "instruct": USE_INSTRUCT_MODEL,
    "emb_dim": selected_cfg["emb_dim"],
    "n_layers": selected_cfg["n_layers"],
    "n_heads": selected_cfg["n_heads"],
    "n_kv_heads": selected_cfg["n_kv_heads"],
    "global_head_dim": selected_cfg["global_head_dim"],
    "num_kv_shared_layers": selected_cfg["num_kv_shared_layers"],
    "device": str(device),
    "dtype": str(model_dtype),
}

3. Load pretrained weights

In [ ]:
# ===== 第3部分：加载预训练权重 =====
# 把从HuggingFace下载的safetensors权重字典(params，键名类似
# "model.language_model.layers.0.self_attn.q_proj.weight")
# 逐一拷贝进我们自己实现的Gemma4DenseModel各个子模块参数里。
def load_weights_into_gemma4_dense(model, cfg, params):
    # 内部工具函数：把right张量的数值拷贝进left参数里，前提是形状必须一致；
    # 用torch.no_grad()包裹以避免被autograd追踪。
    def assign(left, right, tensor_name="unknown"):
        if right is None:
            return False
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor {tensor_name!r}. Left: {tuple(left.shape)}, Right: {tuple(right.shape)}"
            )
        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right.to(dtype=left.dtype, device=left.device))
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))
        return True

    # 不同来源的checkpoint权重键名前缀可能不同
    # （例如是否带"model.language_model."等前缀），这里自动探测params中实际使用的前缀，后面按需拼接。
    prefixes = []
    for prefix in ("model.language_model.", "language_model.", "model.", ""):
        if any(name.startswith(prefix) for name in params):
            prefixes.append(prefix)
    if not prefixes:
        prefixes = [""]

    # 内部工具函数：按候选名称列表(names)依次尝试直接查找，再尝试加各种前缀查找，
    # 找到就返回(tensor, 实际使用的key)，找不到返回(None, None)。
    def get_tensor(*names):
        for name in names:
            if name in params:
                return params[name], name
        for prefix in prefixes:
            for name in names:
                key = f"{prefix}{name}"
                if key in params:
                    return params[key], key
        return None, None

    loaded = 0
    missing = []

    # 内部工具函数：把assign_from(target, 候选名...)封装成"查找+校验形状+拷贝"一条龙，
    # 找不到就记入missing列表，成功则loaded计数+1。
    def assign_from(target, *names):
        nonlocal loaded
        tensor, name = get_tensor(*names)
        expected_name = names[0]
        if tensor is None:
            missing.append(expected_name)
            return
        loaded += int(assign(target, tensor, name or expected_name))

    # 词嵌入表。
    assign_from(model.tok_emb.weight, "embed_tokens.weight")

    # 每层输入(per-layer input)相关的模型级参数(仅当该配置启用时才存在)。
    if getattr(model, "hidden_size_per_layer_input", 0):
        assign_from(model.embed_tokens_per_layer.weight, "embed_tokens_per_layer.weight")
        assign_from(model.per_layer_model_projection.weight, "per_layer_model_projection.weight")
        assign_from(model.per_layer_projection_norm.weight, "per_layer_projection_norm.weight")

    # 逐层加载：每一层都用统一前缀 layers.{layer_idx}. 去拼接HuggingFace的参数名。
    for layer_idx in range(cfg["n_layers"]):
        block = model.blocks[layer_idx]
        prefix = f"layers.{layer_idx}."

        # 注意力部分：Q/K/V/O四个投影权重，以及QK-Norm的权重。
        assign_from(block.att.q_proj.weight, f"{prefix}self_attn.q_proj.weight")
        assign_from(block.att.k_proj.weight, f"{prefix}self_attn.k_proj.weight")
        assign_from(block.att.v_proj.weight, f"{prefix}self_attn.v_proj.weight")
        assign_from(block.att.o_proj.weight, f"{prefix}self_attn.o_proj.weight")
        assign_from(block.att.q_norm.weight, f"{prefix}self_attn.q_norm.weight")
        assign_from(block.att.k_norm.weight, f"{prefix}self_attn.k_norm.weight")

        # 前馈网络(MLP/GeGLU)部分的三个投影权重。
        assign_from(block.mlp.gate_proj.weight, f"{prefix}mlp.gate_proj.weight")
        assign_from(block.mlp.up_proj.weight, f"{prefix}mlp.up_proj.weight")
        assign_from(block.mlp.down_proj.weight, f"{prefix}mlp.down_proj.weight")

        # 四个RMSNorm的权重(注意力前/后、前馈前/后)。
        assign_from(block.input_layernorm.weight, f"{prefix}input_layernorm.weight")
        assign_from(block.post_attention_layernorm.weight, f"{prefix}post_attention_layernorm.weight")
        assign_from(block.pre_feedforward_layernorm.weight, f"{prefix}pre_feedforward_layernorm.weight")
        assign_from(block.post_feedforward_layernorm.weight, f"{prefix}post_feedforward_layernorm.weight")

        # 每层输入门控与投影相关的权重(仅当该层启用per-layer input时才存在)。
        if getattr(block, "hidden_size_per_layer_input", 0):
            assign_from(block.per_layer_input_gate.weight, f"{prefix}per_layer_input_gate.weight")
            assign_from(block.per_layer_projection.weight, f"{prefix}per_layer_projection.weight")
            assign_from(block.post_per_layer_input_norm.weight, f"{prefix}post_per_layer_input_norm.weight")

        # 该层输出的整体缩放标量。
        assign_from(block.layer_scalar, f"{prefix}layer_scalar")

    # 最终归一化层与输出头(若权重绑定，lm_head.weight可能不存在，回退用embed_tokens.weight)。
    assign_from(model.final_norm.weight, "norm.weight")
    assign_from(model.out_head.weight, "lm_head.weight", "embed_tokens.weight")

    # 校验：若一个张量都没加载成功，说明前缀/键名完全对不上，直接报错，
    # 避免"看似加载成功但实际是随机初始化权重"的静默错误。
    if loaded == 0:
        raise KeyError(
            "No Gemma 4 language-model weights were loaded. Supported prefixes are "
            "'model.language_model.', 'language_model.', 'model.', and ''."
        )

    # 校验：若有任何必需的张量缺失，同样直接报错并展示前10个缺失键名，方便排查。
    if missing:
        missing_preview = ", ".join(repr(name) for name in missing[:10])
        if len(missing) > 10:
            missing_preview += f", ... (+{len(missing) - 10} more)"
        raise KeyError(
            f"Missing {len(missing)} required Gemma 4 language-model tensors. "
            f"First missing tensors: {missing_preview}"
        )

    return loaded

# 对外别名，兼容以"load_weights_into_gemma4"命名调用该函数的代码。
load_weights_into_gemma4 = load_weights_into_gemma4_dense

In [ ]:
# Uncomment and run the following code if you are executing the notebook for the first time

# from huggingface_hub import login
# login()
# 首次运行本notebook时，需要先用 huggingface_hub.login() 登录
# (部分Gemma模型需要先在HF网站同意许可协议才能下载)。
# 下面这几行导入是实际下载/加载权重要用到的库：
import json
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

# 根据所选模型规模与是否使用指令微调版本，拼出对应的HuggingFace仓库id，
# 例如 google/gemma-4-E2B-it (指令微调版) 或 google/gemma-4-E2B (基础版)。
repo_id = f"google/gemma-4-{CHOOSE_MODEL}-it" if USE_INSTRUCT_MODEL else f"google/gemma-4-{CHOOSE_MODEL}"
local_dir_name = Path(repo_id).parts[-1]


# 在几个可能的本地目录候选中查找是否已经下载过模型文件，避免重复下载。
def resolve_local_model_dir(local_dir_name):
    candidates = [
        Path(local_dir_name),
        Path("17_gemma4") / local_dir_name,
        Path("ch05") / "17_gemma4" / local_dir_name,
    ]
    for candidate in candidates:
        if (candidate / "model.safetensors").exists() or (candidate / "tokenizer.json").exists():
            return candidate
    return Path(local_dir_name)


# 用真实的运行dtype(model_dtype)构建配置并实例化模型结构(此时参数仍是随机初始化的)。
local_dir = resolve_local_model_dir(local_dir_name)
model_cfg = get_gemma4_dense_config(CHOOSE_MODEL, dtype=model_dtype)
model = Gemma4DenseModel(model_cfg)

# 优先尝试单文件safetensors(E2B/E4B这类较小模型通常是单文件权重)。
single_file = Path(local_dir) / "model.safetensors"

# 若本地已有完整的单文件权重，直接加载；否则先尝试从Hub下载单文件，
# 若单文件也不存在(大模型被切成多个分片)，则退回下载完整仓库快照，
# 按index.json里的weight_map把所有分片safetensors依次加载并合并成一个字典。
if single_file.exists():
    weights_dict = load_file(single_file)
else:
    try:
        weights_path = hf_hub_download(
            repo_id=repo_id,
            filename="model.safetensors",
            local_dir=str(local_dir),
        )
        weights_dict = load_file(weights_path)
    except Exception:
        repo_dir = snapshot_download(repo_id=repo_id, local_dir=str(local_dir))
        index_path = Path(repo_dir) / "model.safetensors.index.json"
        with open(index_path, "r") as f:
            index = json.load(f)

        weights_dict = {}
        for filename in sorted(set(index["weight_map"].values())):
            shard = load_file(Path(repo_dir) / filename)
            weights_dict.update(shard)

# 把下载好的权重字典实际拷贝进模型参数里(调用第3部分定义的函数)。
num_loaded_tensors = load_weights_into_gemma4_dense(model, model_cfg, weights_dict)
print(f"Using Gemma 4 files from: {local_dir}")
print(f"Loaded {num_loaded_tensors} Gemma 4 text tensors")

# 把模型搬到目标计算设备(GPU/MPS/CPU)。
model.to(device)

# 权重已拷贝进模型参数，原始的大字典可以释放以节省内存。
del weights_dict

# 切换到eval模式，关闭dropout等训练专属行为。
model.eval()

In [ ]:
# ===== 第4部分：分词器(Tokenizer) =====
# 基于HuggingFace tokenizers库封装Gemma专用的分词器，
# 并实现Gemma的对话模板(chat template)格式化逻辑。
from tokenizers import Tokenizer


# 轻量封装：底层用tokenizers.Tokenizer加载tokenizer.json，
# 并暴露Gemma特有的几个特殊token(bos/eos/pad/turn)。
class GemmaTokenizer:
    def __init__(self, tokenizer_file_path):
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))

        # Gemma的几个特殊token字符串：
        # <bos>开始符, <eos>结束符, <pad>填充符, <turn|>用于分隔对话轮次的标记。
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.pad_token = "<pad>"
        self.turn_token = "<turn|>"

        # 预先查出各特殊token对应的id，避免每次使用时重复查表。
        self.bos_token_id = self._tok.token_to_id(self.bos_token)
        self.eos_token_id = self._tok.token_to_id(self.eos_token)
        self.pad_token_id = self._tok.token_to_id(self.pad_token)
        self.turn_token_id = self._tok.token_to_id(self.turn_token)

    # 把文本编码成token id列表。
    def encode(self, text, add_special_tokens=True):
        return self._tok.encode(text, add_special_tokens=add_special_tokens).ids

    # 把token id列表(或单个int)解码回文本。
    def decode(self, ids, skip_special_tokens=False):
        if isinstance(ids, int):
            ids = [ids]
        return self._tok.decode(ids, skip_special_tokens=skip_special_tokens)

    # 按Gemma的对话模板拼接多轮对话：
    # <bos><turn|>user\n{内容}<turn|>\n<turn|>model\n{内容}<turn|>\n...
    # add_generation_prompt=True时，末尾追加"<turn|>model\n"提示模型开始生成回复。
    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=False):
        text = self.bos_token
        for message in messages:
            role = "model" if message["role"] == "assistant" else message["role"]
            text += f"{self.turn_token}{role}\n{message['content']}{self.turn_token}\n"

        if add_generation_prompt:
            text += f"{self.turn_token}model\n"

        if tokenize:
            return self.encode(text, add_special_tokens=False)
        return text
# 定位/下载tokenizer.json文件。
tokenizer_file_path = Path(local_dir) / "tokenizer.json"
if not tokenizer_file_path.exists():
    try:
        tokenizer_file_path = Path(
            hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=str(local_dir))
        )
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")

# 实例化分词器，并准备一个示例prompt。
tokenizer = GemmaTokenizer(tokenizer_file_path=str(tokenizer_file_path))
prompt = "Give me a short introduction to large language models."

# 指令微调模型需要套用对话模板并生成"生成提示"后缀；
# 基础(非instruct)模型则用简单的"问答"格式拼接。
if USE_INSTRUCT_MODEL:
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    input_token_ids = tokenizer.encode(prompt, add_special_tokens=False)
else:
    prompt = f"{prompt}\n\nAnswer:"
    input_token_ids = tokenizer.encode(prompt)

# 这是一个未赋值的表达式，在Jupyter中会自动把解码后的prompt文本作为该cell输出显示，
# 用来直观检查分词/模板拼接是否符合预期。
tokenizer.decode(input_token_ids, skip_special_tokens=False)

5. Generate text

In [ ]:
# ===== 第5部分：文本生成 =====
# 用贪心解码(greedy decoding，每步取概率最大的token)逐token生成，
# 并以生成器(yield)方式流式输出，边生成边打印。
# Optionally use torch.compile for an extra speed-up
# model = torch.compile(model)
# token_ids: (batch_size, 当前序列长度)。每步都对整段序列做前向计算(未实现KV cache)，
# 但通过reuse_shared_kv=True复用模型内部"跨层KV共享"的优化路径来减少部分重复计算。
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()

    # 推理时关闭梯度计算，节省显存并加速。
    with torch.no_grad():
        for _ in range(max_new_tokens):
            try:
                # 优先尝试开启跨层KV共享(reuse_shared_kv=True)；若模型的forward不支持
                # 该关键字参数(抛出TypeError)，则退回不带该参数的调用方式做兼容。
                # 取序列最后一个位置的logits: (batch, vocab_size)。
                out = model(token_ids, reuse_shared_kv=True)[:, -1]
            except TypeError:
                out = model(token_ids)[:, -1]
            # 贪心解码：直接选取logits最大的token作为下一个token。
            next_token = torch.argmax(out, dim=-1, keepdim=True)

            # 命中结束符(instruct模型用turn_token表示一轮结束，否则用eos_token)则停止生成。
            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break

            # 把新生成的token通过生成器yield出去，交由外层循环即时打印。
            yield next_token
            # 把新token拼接到已有序列末尾，作为下一步的输入
            # (未使用KV cache，每步都重新过一遍完整序列，仅通过跨层共享K/V降低部分重复计算)。
            token_ids = torch.cat([token_ids, next_token], dim=1)
# 把prompt的token id列表转换成模型输入张量，形状(1, prompt_len)，batch_size=1。
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

# 若在GPU上运行，重置显存峰值统计，便于之后测量本次生成实际占用的显存。
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# 指令微调模型遇到<turn|>就停止(表示模型认为自己这一轮回复说完了)；
# 基础模型则以标准的<eos>作为停止符。
stop_token_id = tokenizer.turn_token_id if USE_INSTRUCT_MODEL else tokenizer.eos_token_id

# 流式生成最多200个新token，边生成边解码打印。
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=200,
    eos_token_id=stop_token_id,
):
    # 把生成的单个token张量转换成python list，再解码成文本片段。
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)

# 生成结束后，若在GPU上运行，则打印本次生成过程中使用的峰值显存(GB)。
if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")